# 04 — Triple Exponential Smoothing (Holt-Winters)

**Reference:** Vandeput, N. (2021). *Data Science for Supply Chain Forecasting* (2nd ed.), Chapters 9 & 11.

Triple exponential smoothing (Holt-Winters) adds a **seasonality component** to Holt's method.
This makes it the most complete classical model — capturing level, trend, and seasonality.

Two variants:
- **Multiplicative** (Chapter 9): seasonality scales with the level — common in FMCG
- **Additive** (Chapter 11): seasonality is a fixed offset — better for stable seasonal swings

**For energy drinks:** seasonality is multiplicative — summer sales are a *multiple* of winter sales,
not a fixed addition. This is the Red Bull-relevant model.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

## 1. Simulate Seasonal FMCG Demand

Energy drink demand — strong summer peak, dip in winter. Monthly, 4 years of history.

In [ ]:
np.random.seed(42)
n_years = 4
# Seasonal indices: summer peak in months 6-8, trough in Jan-Feb
seasonal_pattern = np.array([
    0.75, 0.78, 0.88, 0.95, 1.05, 1.20,
    1.30, 1.25, 1.10, 0.98, 0.85, 0.80
])
base_demand = 1200
trend_per_month = 8

demand = []
for year in range(n_years):
    for month in range(12):
        t = year * 12 + month
        d = (base_demand + trend_per_month * t) * seasonal_pattern[month]
        d += np.random.normal(0, 40)
        demand.append(max(0, d))

demand = np.array(demand)
print(f'Periods: {len(demand)} months ({n_years} years)')
print(f'Mean demand: {demand.mean():.0f} | Min: {demand.min():.0f} | Max: {demand.max():.0f}')

## 2. Multiplicative Holt-Winters Model

In [ ]:
def holt_winters_multiplicative(d, slen=12, extra_periods=12,
                                 alpha=0.4, beta=0.2, gamma=0.4):
    """
    Multiplicative Holt-Winters seasonal exponential smoothing.

    Parameters
    ----------
    d      : historical demand
    slen   : seasonal period length (12 for monthly)
    alpha  : level smoothing
    beta   : trend smoothing
    gamma  : seasonal smoothing
    """
    cols = len(d)
    d = np.append(d, [np.nan] * extra_periods)
    f  = np.full(cols + extra_periods, np.nan)
    l_ = np.full(cols + extra_periods, np.nan)
    b_ = np.full(cols + extra_periods, np.nan)
    s_ = np.full(cols + extra_periods + slen, np.nan)

    # Initialise level, trend, and seasonal indices from first year
    l_[0] = np.mean(d[:slen])
    b_[0] = (np.mean(d[slen:2*slen]) - np.mean(d[:slen])) / slen
    for i in range(slen):
        s_[i] = d[i] / l_[0]

    for t in range(1, cols):
        f[t] = (l_[t-1] + b_[t-1]) * s_[t-1]
        l_[t] = alpha * (d[t] / s_[t-1]) + (1 - alpha) * (l_[t-1] + b_[t-1])
        b_[t] = beta * (l_[t] - l_[t-1]) + (1 - beta) * b_[t-1]
        s_[t + slen - 1] = gamma * (d[t] / l_[t]) + (1 - gamma) * s_[t-1]

    for m in range(1, extra_periods + 1):
        f[cols + m - 1] = (l_[cols-1] + m * b_[cols-1]) * s_[cols + m - 2]

    return pd.DataFrame({'Demand': d, 'Forecast': f, 'Error': d - f})


df_hw = holt_winters_multiplicative(demand, slen=12, extra_periods=12)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df_hw['Demand'], label='Actual Demand', color='#2C2C2A', linewidth=1.5)
ax.plot(df_hw['Forecast'], label='Holt-Winters Forecast', color='#185FA5',
        linestyle='--', linewidth=1.5)
ax.axvline(x=len(demand), color='gray', linestyle=':', alpha=0.5,
           label='Forecast horizon')
ax.set_title('Holt-Winters Multiplicative — Level + Trend + Seasonality', fontsize=13)
ax.set_xlabel('Period (months)')
ax.set_ylabel('Demand (units)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Seasonal Indices — What the Model Learned

In [ ]:
months = ['Jan','Feb','Mar','Apr','May','Jun',
          'Jul','Aug','Sep','Oct','Nov','Dec']
# True seasonal pattern vs what a naive first-year estimate gives
naive_seasonal = np.array([demand[m::12].mean() for m in range(12)])
naive_seasonal = naive_seasonal / naive_seasonal.mean()

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(12)
ax.bar(x - 0.2, seasonal_pattern, 0.4, label='True pattern', color='#185FA5', alpha=0.7)
ax.bar(x + 0.2, naive_seasonal, 0.4, label='Estimated from data', color='#D85A30', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(months)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Seasonal Indices — Energy Drink Demand Pattern', fontsize=13)
ax.set_ylabel('Seasonal Index (1.0 = average)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

Holt-Winters is the **benchmark model** for seasonal FMCG demand forecasting.
Before applying any ML model, always check if Holt-Winters is already good enough —
it often is for products with stable seasonal patterns.

| Component | Controlled by | What it captures |
|-----------|--------------|------------------|
| Level | α | Current demand level |
| Trend | β | Month-over-month growth |
| Seasonality | γ | Repeating yearly pattern |

**Next:** Outlier handling (`05_outlier_handling.ipynb`) before moving to ML.
